# 03 — `Protocol` vs `ABC` (structural vs nominal typing)

We met `abc.ABC` in folder 04 — declare a contract, force subclasses to implement it. That's **nominal** typing: "only things that *say they are X* (by inheritance) count as X."

**`Protocol`** is the alternative: **structural** typing — "anything with the right methods counts." Duck typing, but checked. This is what unlocks clean DI: your service depends on a *shape*, not on a specific class hierarchy.

Spoiler: **prefer `Protocol` for interfaces in new code.**

## The same example, both ways

In [1]:
from abc import ABC, abstractmethod

# ABC version — subclasses must INHERIT.
class Greeter(ABC):
    @abstractmethod
    def greet(self, name: str) -> str: ...

class FormalGreeter(Greeter):
    def greet(self, name: str) -> str:
        return f"Good day, {name}."

# This class has the same shape but does NOT inherit from Greeter:
class CasualGreeter:
    def greet(self, name: str) -> str:
        return f"hey {name}!"

def welcome(g: Greeter, name: str) -> None:
    print(g.greet(name))

welcome(FormalGreeter(), "Alice")
welcome(CasualGreeter(), "Bob")     # runtime OK, but mypy: error — CasualGreeter is not a Greeter
# isinstance check would fail too:
print(isinstance(CasualGreeter(), Greeter))

Good day, Alice.
hey Bob!
False


In [2]:
from typing import Protocol

# Protocol version — no inheritance required. Any class with `greet(str) -> str` qualifies.
class Greeter(Protocol):
    def greet(self, name: str) -> str: ...

class FormalGreeter:
    def greet(self, name: str) -> str:
        return f"Good day, {name}."

class CasualGreeter:
    def greet(self, name: str) -> str:
        return f"hey {name}!"

def welcome(g: Greeter, name: str) -> None:
    print(g.greet(name))

welcome(FormalGreeter(), "Alice")     # mypy happy
welcome(CasualGreeter(), "Bob")       # mypy happy too — structural match

Good day, Alice.
hey Bob!


## Why Protocol wins for interfaces

Imagine you're calling a third-party class you can't modify. With an ABC, you'd have to wrap or subclass it. With a Protocol, you don't have to do anything — if the class has the methods, it works.

It's also the natural fit for *dependency injection*: a service says "I need something that can `find_by_id` and `save`" — without coupling to a specific DB-backed implementation. We do this for real in [04_repository_protocol/](04_repository_protocol/).

## `@runtime_checkable` — when you want `isinstance` to work

By default, `Protocol` is a *type-checker-only* concept — `isinstance(x, MyProtocol)` raises. Add `@runtime_checkable` and it works (it just checks method names, not signatures).

In [3]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

class Circle:
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

class Empty: pass

print(isinstance(Circle(3), HasArea))   # True
print(isinstance(Empty(),   HasArea))   # False

True
False


Use `@runtime_checkable` sparingly. The point of Protocol is *static* checking; reaching for `isinstance` often signals the design wants polymorphism instead. If you find yourself writing `if isinstance(x, HasArea): x.area()`, just call `x.area()` and let the type-checker yell.

## Generic protocols

Combine the last two notebooks: a generic, structural interface. This is the shape of `Repository[T]` we'll build next.

In [4]:
from typing import Protocol

class Repository[T](Protocol):
    def find_by_id(self, id: int) -> T | None: ...
    def save(self, item: T) -> None: ...
    def all(self) -> list[T]: ...

# Any class with those three methods (on the right T) satisfies Repository[T] structurally.
# We'll build a real in-memory implementation in the next file.

## `typing.Annotated` — bolting metadata onto a type

`Annotated[T, ...stuff...]` keeps the type as `T` but carries extra information for tools to read. **FastAPI and Pydantic use this everywhere** — `Annotated[int, Field(gt=0)]`, `Annotated[User, Depends(get_current_user)]`.

In [5]:
from typing import Annotated, get_type_hints

# A toy decorator that reads validation metadata from Annotated.
def validate(func):
    hints = get_type_hints(func, include_extras=True)
    def wrapper(*args, **kwargs):
        bound = dict(zip(hints, args))
        bound.update(kwargs)
        for name, hint in hints.items():
            if name == "return":
                continue
            # Annotated[int, "positive"] has metadata in hint.__metadata__
            for tag in getattr(hint, "__metadata__", ()):
                if tag == "positive" and bound[name] <= 0:
                    raise ValueError(f"{name}={bound[name]} must be > 0")
        return func(*args, **kwargs)
    return wrapper

@validate
def take_dose(amount: Annotated[int, "positive"]) -> str:
    return f"took {amount} mg"

print(take_dose(50))
try:
    take_dose(-1)
except ValueError as e:
    print("rejected:", e)

took 50 mg
rejected: amount=-1 must be > 0


**Pattern to internalize:** the type system tells *callers* what to pass; the metadata tells *libraries* (FastAPI, Pydantic, validators) what extra behavior to wire up. One annotation, two audiences.

## When to use ABC vs Protocol

| Use **ABC** | Use **Protocol** |
|---|---|
| You also want **shared concrete code** (the template-method pattern) | You only want to declare an **interface** |
| You want `isinstance(x, Base)` checks (instance of a real class) | You want any duck-typed class to qualify |
| Your subclasses naturally form a hierarchy | Implementations are unrelated to each other (different libraries, mocks for tests) |

**For dependency injection: Protocol almost always.** It's why FastAPI's `Depends` cares about the *callable's return type*, not its class hierarchy.

## Mini-exercise

1. Define a `Comparable` Protocol with a `__lt__` method. Write `min_of[T: Comparable](a: T, b: T) -> T`.
2. Take the `Shape` ABC from [04_oop_advanced/04_shapes_app/shapes.py](../04_oop_advanced/04_shapes_app/shapes.py). Rewrite it as a `Protocol`. Could `Canvas` accept ANY object with `area()` and `perimeter()` now? Why is that better or worse?
3. Annotate a parameter with `Annotated[str, "non-empty"]` and write a tiny validator that reads the metadata and rejects empty strings.

In [6]:
# your solutions here


**Takeaways**

- ABC = **nominal**: "is-a-subclass-of". Protocol = **structural**: "has-these-methods".
- Prefer `Protocol` for interfaces; use ABC when you also want shared concrete code.
- Protocols enable clean DI — your code depends on a *shape*, not a class.
- `Annotated[T, ...]` keeps the type but adds metadata libraries can read — the foundation of FastAPI `Depends` and Pydantic `Field`.

Next: [04_repository_protocol/](04_repository_protocol/) — Protocol + DI for real.